# 40 TGN: convert, train, infer (tasks S1.7 / S4.2)

Thin Colab driver: all logic lives in `citecurv.tgn` (`convert.to_jodie`, `train.train_tgn`, `infer.infer_tgn`) and in the vendored DyGLib under `third_party/DyGLib` (patches in its `PATCHES.md`). Runtime: GPU (T4 is enough for the pilot slice).

Inputs on Drive: the contract `events` frame (`event_id, src_id, dst_id, t`) and `embeddings.npz` (`ids`, `vectors`, SPECTER2, d = 768). Outputs on Drive: the run folder (checkpoint, `metrics.json`, `resolved_args.json`) and `tgn_event_scores.parquet` + `tgn_event_embeddings.npz`.

Session safety: DyGLib has no mid-run resume. The run folder is written to local disk and copied to Drive at the end of each step; a session that dies during training loses that training run. Long runs are chunked by seed (PROVISIONAL(G1)).

In [ ]:
# 1. Versions. The project pins Python 3.12 (configs/pins.md); Colab images may ship a newer Python.
import sys, platform, warnings
print('python', platform.python_version())
if sys.version_info[:2] != (3, 12):
    warnings.warn(f'Python {platform.python_version()} is not the pinned 3.12; results may not reproduce. '
                  'Record this in PROGRESS.md or switch the runtime version.')
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.__version__.split('.')[0] == '2', 'the tgn stage pins torch 2.x'
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Install citecurv from a repository tag. DyGLib is vendored in the repository (not in the wheel),
#    so clone the tag and install from the clone.
REPO_URL = 'https://github.com/Natboa/Detecting-Network-Citation-Anomalies-Using-Graph-Curvature.git'
REPO_TAG = 'TAG-PLACEHOLDER'  # set to the release tag that contains this notebook's code
REPO_DIR = '/content/citecurv'
!rm -rf {REPO_DIR} && git clone --depth 1 --branch {REPO_TAG} {REPO_URL} {REPO_DIR}
# tgn stage pins (configs/pins.md): torch 2.x (Colab's preinstalled CUDA build), no PyG.
# DyGLib itself needs numpy, pandas, tqdm, scikit-learn, all preinstalled on Colab.
!pip install -q {REPO_DIR} pyyaml
import os
os.environ['CITECURV_DYGLIB_ROOT'] = f'{REPO_DIR}/third_party/DyGLib'

In [ ]:
# 3. Drive and run settings. Values marked SET come from configs/default.yaml once merged (tgn block).
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/citecurv')
EVENTS_PATH = DRIVE / 'events.parquet'      # SET: contract events frame (event_id, src_id, dst_id, t)
EMB_PATH = DRIVE / 'embeddings.npz'         # SET: SPECTER2 ids/vectors
NAME = 'pilot'                              # SET: dataset name (DyGLib --dataset_name)
SEED = 0                                    # one of seed_list [0, 1, 2, 3, 4]
EPOCHS = 5                                  # SET: pilot; S4.2 uses the configured value
LEARNING_RATE = 1e-4                        # DyGLib default
TRAIN_END_T = None                          # SET: last training day (days since epoch) from the S4.1 split
VAL_END_T = None                            # SET: last validation day
LEAK_CHECK = True                           # S1.7: run the memory leak hook during training (O(nodes) per batch)
EMB_DTYPE = 'float32'                       # 'float16' halves tgn_event_embeddings.npz
LOCAL = Path('/content/tgn')
DATA_DIR = LOCAL / 'data'
RUN_DIR = LOCAL / 'runs' / f'TGN_{NAME}_seed{SEED}'
DRIVE_RUN = DRIVE / 'runs' / f'TGN_{NAME}_seed{SEED}'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE, RUN_DIR, DRIVE_RUN)

In [ ]:
# 4. Convert to DyGLib's JODIE format (local disk).
import numpy as np, pandas as pd
from citecurv.tgn.convert import to_jodie
events = pd.read_parquet(EVENTS_PATH)
with np.load(EMB_PATH, allow_pickle=False) as z:
    emb = (z['ids'], z['vectors'])
meta = to_jodie(events, emb, DATA_DIR, NAME)
print(meta)
n, d = meta['n_nodes'], meta['node_feat_dim']
print(f'memory matrix: {n} nodes x {d} floats = {n * d * 4 / 2**30:.2f} GB (float32), held twice per batch by DyGLib')

In [ ]:
# 5. Train. DyGLib's split boundaries are put on the project's dates so the encoder never trains on
#    validation or test events (citecurv.tgn.train.dyglib_split_ratios).
import json, shutil
from citecurv.tgn.train import train_tgn, dyglib_split_ratios
if TRAIN_END_T is not None and VAL_END_T is not None:
    val_ratio, test_ratio = dyglib_split_ratios(events['t'].to_numpy(), TRAIN_END_T, VAL_END_T)
else:
    val_ratio, test_ratio = 0.15, 0.15
    print('WARNING: no split dates set; using DyGLib quantile split 0.15/0.15 (pilot only)')
metrics = train_tgn(DATA_DIR, NAME, epochs=EPOCHS, seed=SEED, device=DEVICE, node_feat_dim=meta['node_feat_dim'],
                    out_dir=RUN_DIR, leak_check=LEAK_CHECK, learning_rate=LEARNING_RATE,
                    val_ratio=val_ratio, test_ratio=test_ratio)
print(pd.DataFrame(metrics['epochs']).to_string(index=False))
print('test', metrics['test'], '| leak check', metrics['leak_check'])
print('peak RSS MB', metrics['peak_rss_mb'], '| CUDA peak MB', metrics['cuda_peak_mb'], '| wall s', metrics['wall_clock_s'])
shutil.copytree(RUN_DIR, DRIVE_RUN, dirs_exist_ok=True, ignore=shutil.ignore_patterns('dyglib_work'))

In [ ]:
# 6. Infer: p_link / neg_p_link and endpoint embeddings for every event, memory replayed from empty.
from citecurv.tgn.infer import infer_tgn
scores, emb_file = infer_tgn(RUN_DIR, DATA_DIR, NAME, out_dir=RUN_DIR / 'infer', seed=SEED, device=DEVICE,
                             embeddings_dtype=EMB_DTYPE)
print(scores.describe())
print('share of events with p_link > neg_p_link:', float((scores.p_link > scores.neg_p_link).mean()))

In [ ]:
# 7. Save to Drive (few large files) and print what landed there.
shutil.copytree(RUN_DIR / 'infer', DRIVE_RUN / 'infer', dirs_exist_ok=True)
for p in sorted(DRIVE_RUN.rglob('*')):
    if p.is_file():
        print(f'{p.stat().st_size / 2**20:10.1f} MB  {p.relative_to(DRIVE)}')